# TP4 - Problem Settings and Data Generation

## System of Poisson equations

This notebook defines the **problem settings** for **Test Problem 4 (TP4)**, which addresses a **system of Poisson equations** on the **3D column geometry**.

The purpose of this notebook is to:

*   define the physical problem and its parameters,

*   generate **simulated IoT-like boundary measurements**,

*   acquire and preprocess the 3D geometry,

*   produce the mesh and visualization-ready files required by the subsequent stage.

This notebook represents the **first stage of the TP4 pipeline** and prepares all the assets used in the **Direct Problem Submodule**.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import DRT_PATH, TP4_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + DRT_PATH + TP4_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"

In [ ]:
# Create needed folders
import os

lst_folders = ["figures", "files", "models"]

for name in lst_folders:
    os.makedirs(os.path.join(ABS_PATH, name), exist_ok=True)

## Libraries and Dependencies

We start by importing all the libraries required for:
- geometry handling and mesh generation,
- data management,
- preparation of visualization files.

In [ ]:
import torch
import pandas as pd

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

## Numerical Precision

Double precision is enforced

In [ ]:
torch.set_default_dtype(torch.float64)

## Physical Problem: system of Poisson equations

We consider a 3D domain $\Omega \subseteq \mathbb{R}^3$ representing a **column geometry**, with boundary $\Gamma = \partial \Omega$.  

The physical phenomenon is described by the following differenzial problem:

\begin{cases}
    \Delta u (x, y, z ) = F(x, y, z, u) & \Omega \\
    u (x, y, z) = B (x, y, z) & \partial \Omega
    \tag{1}
\end{cases}
where $u = (u_1, u_2)^T \in \mathbb{R}^2$,

\begin{equation}
    F (x, y, z, u) = \left(
        \begin{array}{c}
            2 u_1 \\
            2
        \end{array}
    \right), \qquad (x, y, z ) \in \Omega ,
    \tag{2}
\end{equation}

\begin{equation}
    B (x, y, z) = \left(
        \begin{array}{c}
            e^{x+y} \\
            x^2 - z
        \end{array}
    \right), \qquad (x, y, z ) \in \partial \Omega .
    \tag{2}
\end{equation}

The analytical solution is consistent with boundary conditions.

# Generation of Data for boundary conditions

In this stage, we generate **simulated measurements** that emulate IoT sensor data.

Sensors are assumed to be located on the boundary of the 3D domain.  
The corresponding values of the physical field are computed using the analytical solution and stored.

Sampling of collocation points and boundary points.

In [ ]:
column = Blend2Pina(LOAD_MODEL + model_name)

num_points_int = 10_000
num_points_b = 10_000

domain = column.intern()
surface = column.boundary()
points_internal = domain.sample(num_points_int)
points_boundary = surface.sample(num_points_b)

Storage of sampled points.

In [ ]:
df_int = pd.DataFrame(
    points_internal.tensor.detach().numpy()
)
df_int.to_csv("./files/input_int.csv", sep = ";")

In [ ]:
u1 = torch.exp(points_boundary.extract('x') + points_boundary.extract('y'))
u2 = (points_boundary.extract('x')**2 - points_boundary.extract('z'))

In [ ]:
total_info = torch.concat(
    [points_boundary.tensor, u1, u2],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "u1", "u2"]
)

df.to_csv("./files/data.csv", sep=";")

## Mesh Generation and Visualization Files

The 3D column geometry is discretized to generate a computational mesh suitable for numerical simulations.

In addition:
- visualization-ready files (.xdmf and associated data) are produced,
- these files enable inspection of solutions and errors in ParaView,
- the same mesh is reused consistently across all TP1 stages.

In [ ]:
column_msh = Blend2Mesh(LOAD_MODEL + model_name, "column")

In [ ]:
column_msh.create_single_meshes(len_msh=.01)

In [ ]:
column_xdmf = Msh2Xdmf("column.msh", "column")
column_xdmf.to_xdmf(num_refine=3)
column_xdmf.to_xdmf()